<table dir="ltr" width="100%"><tr><td width="50%" valign="top" dir="ltr" lang="en" align="left"><h2>Setup</h2><p>Colab: use CPU, save a copy and run this cell. Local Jupyter: install requirements-day01.txt first. No Docker, API token or GPU.</p></td><td width="50%" valign="top" dir="rtl" lang="ar" align="right"><h2>الإعداد</h2><p>في Colab اختر CPU واحفظ نسخة وشغل هذه الخلية. في Jupyter المحلي ثبت requirements-day01.txt أولًا. لا يلزم Docker أو رمز API أو GPU.</p></td></tr></table>



In [2]:
from pathlib import Path
import os, sys, subprocess, importlib.metadata
IS_COLAB = Path('/content').is_dir() and 'google.colab' in sys.modules
if IS_COLAB:
    ROOT = Path('/content/masar-modern-data-engineering')
    if not ROOT.exists():
        subprocess.run(['git','clone','https://github.com/almiyead-rgb/masar-modern-data-engineering.git',str(ROOT)], check=True)
    pins = {'pyspark':'3.5.8','delta-spark':'3.3.3','py4j':'0.10.9.9'}
    missing = []
    for package, version in pins.items():
        try: observed = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError: observed = None
        if observed != version: missing.append(f'{package}=={version}')
    if missing:
        subprocess.run([sys.executable,'-m','pip','install','--quiet',*missing],check=True)
    candidates = list(Path('/usr/lib/jvm').glob('*17*'))
    if not any((p/'bin/java').is_file() for p in candidates):
        subprocess.run(['apt-get','update','-qq'],check=True)
        subprocess.run(['apt-get','install','-y','-qq','openjdk-17-jre-headless'],check=True)
        candidates = list(Path('/usr/lib/jvm').glob('*17*'))
    os.environ['JAVA_HOME'] = str(next(p for p in candidates if (p/'bin/java').is_file()))
    os.chdir(ROOT)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'course.json').is_file()),None)
    if ROOT is None: raise FileNotFoundError('Open from the repository root; see docs/SETUP.md')
print('Repository:', ROOT)
print('Python:', sys.version.split()[0])


Repository: /content/masar-modern-data-engineering
Python: 3.11.13


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Lab 01 preparation · Inspect the fixed sources</h1><p>This is the source-inspection part, not the complete Bronze lab. The complete lab also requires real Delta writes and replay evidence.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>تحضير اللاب 01 · فحص المصادر الثابتة</h1><p>هذا جزء فحص المصادر لا لاب Bronze كاملًا؛ فاللاب الكامل يتطلب كتابة Delta فعلية وأدلة الإعادة.</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Goal and setup</h2><p>Read the manifest, count the three base feeds and examine keys, relationships and city-label formatting. Keep the whole repository together; shared code is in <code>src/masar/sources.py</code>.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>الهدف والإعداد</h2><p>اقرأ السجل وعد المصادر الثلاثة وافحص المفاتيح والعلاقات وكتابة المدن. احتفظ بالمستودع كاملًا؛ الكود المشترك في <code dir="ltr">src/masar/sources.py</code>.</p></td></tr></tbody></table>

In [3]:
from pathlib import Path
import sys, json, tempfile
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src" / "masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the complete course repository.")
sys.path.insert(0, str(ROOT / "src"))
(ROOT / "outputs").mkdir(exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix="day01_", dir=ROOT / "outputs"))
print("Inspect the source and assumptions before running the next native section.")

Inspect the source and assumptions before running the next native section.


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>1. Verify and load</h2><p>Reject changed source files before interpreting their content. This does not modify the inputs.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>1. التحقق والتحميل</h2><p>ارفض تغير ملفات المصدر قبل تفسير محتواها. لا تغير الخطوة المدخلات.</p></td></tr></tbody></table>

In [4]:
from masar.sources import verify_manifest, load_sources, profile_sources
SOURCE = ROOT / "data" / "masar-small-v1"
manifest = verify_manifest(SOURCE)
feeds = load_sources(SOURCE)
print("Dataset:", manifest["label"])
print("Verified manifest files:", len(manifest["files"]))
print(json.dumps({name: len(rows) for name, rows in feeds.items()}, indent=2))
print("Trip columns:", list(feeds["trips"][0]))
print("First synthetic event:", json.dumps(feeds["gps_events"][0], ensure_ascii=False))

Dataset: MASAR_SMALL_V1
Verified manifest files: 10
{
  "trips": 72,
  "drivers": 6,
  "gps_events": 216
}
Trip columns: ['trip_id', 'driver_id', 'city', 'start_ts', 'end_ts', 'fare_sar', 'distance_km']
First synthetic event: {"city": "Riyadh", "event_id": "SYN_E0001_0", "event_ts": "2026-06-01T06:00:00+03:00", "location": {"lat": 24.7, "lon": 46.7}, "synthetic": true, "trip_id": "SYN_T0001"}


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>2. Profile before cleaning</h2><p>Count keys and inspect relationships. The city-normalization comparison is a view of the inputs, not a rewrite of Bronze.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>2. افحص قبل التنظيف</h2><p>عد المفاتيح وافحص العلاقات. مقارنة توحيد المدن فحص للمدخلات، وليست إعادة كتابة Bronze.</p></td></tr></tbody></table>

In [5]:
result = profile_sources(SOURCE)
print(json.dumps(result["profile"], indent=2))
print(json.dumps(result["relations"], indent=2))
print(json.dumps(result["city_profile"], indent=2))

{
  "trips": {
    "rows": 72,
    "key": "trip_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "city": 0,
      "distance_km": 0,
      "driver_id": 0,
      "end_ts": 0,
      "fare_sar": 0,
      "start_ts": 0,
      "trip_id": 0
    }
  },
  "drivers": {
    "rows": 6,
    "key": "driver_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "driver_id": 0,
      "driver_rating": 0,
      "vehicle_type": 0
    }
  },
  "gps_events": {
    "rows": 216,
    "key": "event_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "city": 0,
      "event_id": 0,
      "event_ts": 0,
      "location": 0,
      "synthetic": 0,
      "trip_id": 0
    }
  }
}
{
  "trips_without_driver": 0,
  "events_without_trip": 0,
  "events_with_invalid_coordinates": 0
}
{
  "original_labels": {
    " dammam ": 3,
    " jeddah ": 3,
    " riyad

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>3. Check and preserve evidence</h2><p>The checks below concern the source files only. A replay into append-only Bronze is a separate operation and has not been run here.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>3. تحقق واحفظ الأدلة</h2><p>تخص الفحوص التالية ملفات المصدر فقط. إعادة الاستيعاب في Bronze بنمط الإضافة عملية منفصلة لم تنفذ هنا.</p></td></tr></tbody></table>

In [6]:
if not all(result["checks"].values()):
    raise AssertionError(result["checks"])
output = RUN / "source_inspection.json"
output.write_text(json.dumps(result, ensure_ascii=False, sort_keys=True, indent=2) + "\n", encoding="utf-8")
print(json.dumps(result["checks"], indent=2))
print("PASS: source inspection only")
print("Saved:", output.name)
print("Source inspection complete. Continue to the Bronze section.")

{
  "source_counts": true,
  "unique_base_keys": true,
  "base_top_level_complete": true,
  "valid_links_and_coordinates": true,
  "three_events_per_trip": true,
  "base_city_set": true
}
PASS: source inspection only
Saved: source_inspection.json
Source inspection complete. Continue to the Bronze section.


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Interpretation and next step</h2><p>Explain why joining each trip to all its events changes the row grain. Explain why a unique base key does not guarantee uniqueness after replay. Return to the Lab 01 contract; its engine steps remain pending.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>التفسير والخطوة التالية</h2><p>اشرح لماذا يغير ربط الرحلة بجميع أحداثها مستوى الصف، ولماذا لا يضمن المفتاح الفريد في المصدر عدم التكرار بعد الإعادة. ارجع إلى مواصفات اللاب 01؛ خطوات محركه ما تزال معلقة.</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Lab 01 · Real Delta Bronze</h1><p>SDA-DSC-214 · Meaad Al-Marri</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>اللاب 01 · Bronze فعلية باستخدام Delta</h1><p>SDA-DSC-214 · ميعاد المري</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Setup · report the actual environment</h2><p>Only inspect dependencies here. This cell does not install packages or start Spark.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>الإعداد · اعرض البيئة الفعلية</h2><p>افحص المتطلبات فقط هنا. لا تثبت هذه الخلية حزمًا ولا تبدأ Spark.</p></td></tr></tbody></table>

In [7]:
from pathlib import Path
import sys, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from within the complete course repository")
sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.runtime import inspect_environment, require_environment, start_spark
print(json.dumps(inspect_environment(), indent=2))

{
  "scope": "DEPENDENCY_PREFLIGHT_ONLY",
  "python": "3.11.13",
  "java": "openjdk version \"17.0.20\" 2026-07-21",
  "java_major": 17,
  "packages": {
    "pyspark": {
      "required": "3.5.8",
      "observed": "3.5.8"
    },
    "delta-spark": {
      "required": "3.3.3",
      "observed": "3.3.3"
    },
    "py4j": {
      "required": "0.10.9.9",
      "observed": "0.10.9.9"
    }
  },
  "status": "DEPENDENCIES_PRESENT_ENGINE_NOT_TESTED",
  "issues": [],
  "engine_executed": false
}


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Start a clean, bounded run</h2><p>Fail clearly if dependencies are missing; never switch engines silently. Existing work is retained.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>ابدأ تشغيلًا جديدًا محدود النطاق</h2><p>توقف بوضوح عند نقص المتطلبات دون تبديل المحرك بصمت. تبقى الأعمال السابقة محفوظة.</p></td></tr></tbody></table>

In [8]:
require_environment()
from masar.workspace import new_workspace, require_fixed_dataset, write_json, workspace_path
require_fixed_dataset(SOURCE)
WORK = new_workspace(ROOT, "day01_bronze")
print("Workspace:", WORK.relative_to(ROOT))

Workspace: outputs/day01_bronze_jmbv27q9


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Read one source</h2><p>CSV strings remain unchanged. The count action starts computation.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>اقرأ مصدرًا واحدًا</h2><p>تبقى نصوص CSV كما هي. يطلق فعل العد الحساب الفعلي.</p></td></tr></tbody></table>

In [9]:
from masar.bronze import raw_frame, ingest_feed, verify_bronze
# Functions are imported without starting Spark. The next cell executes the lab.
print("Bronze functions loaded. The next cell writes and reads the Delta tables.")

Bronze functions loaded. The next cell writes and reads the Delta tables.


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Load, replay, verify and retain evidence</h2><p>The <code>try/finally</code> always stops the session after this block, even on failure. Reuse of a committed batch id is rejected. See <a href="../src/masar/bronze.py">shared source</a>; this is not a placeholder or mock engine.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>استوعب وأعد الوصول وتحقق واحفظ الدليل</h2><p>توقف <code>try/finally</code> الجلسة بعد هذه الكتلة حتى عند الفشل. تُرفض إعادة استخدام معرف دفعة محفوظ. راجع <a href="../src/masar/bronze.py">المصدر المشترك</a>؛ ليست هذه محاكاة للمحرك أو موضعًا فارغًا.</p></td></tr></tbody></table>

In [10]:
from masar.workspace import record_bronze_success
spark = start_spark(WORK)
try:
    print("Spark:", spark.version)
    raw_trips = raw_frame(spark, SOURCE, "trips")
    raw_trips.printSchema()
    raw_trips.show(3, truncate=False)
    print("Source trips:", raw_trips.count())
    for feed in ("trips", "drivers", "gps_events"):
        print(ingest_feed(spark, SOURCE, WORK, feed, "base_001"))
    print(ingest_feed(spark, SOURCE, WORK, "trips", "replay_002"))
    report = verify_bronze(spark, SOURCE, WORK)
    record_bronze_success(ROOT, WORK)
    print(json.dumps({"counts": report["counts"], "checks": report["checks"]}, indent=2))
    print("Retained:", (WORK / "reports/bronze.json").relative_to(ROOT))
finally:
    spark.stop()

Spark: 3.5.8
root
 |-- trip_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- start_ts: string (nullable = true)
 |-- end_ts: string (nullable = true)
 |-- fare_sar: string (nullable = true)
 |-- distance_km: string (nullable = true)

+---------+---------+------+-------------------------+-------------------------+--------+-----------+
|trip_id  |driver_id|city  |start_ts                 |end_ts                   |fare_sar|distance_km|
+---------+---------+------+-------------------------+-------------------------+--------+-----------+
|SYN_T0001|SYN_D001 |Riyadh|2026-06-01T06:00:00+03:00|2026-06-01T06:08:00+03:00|18.00   |3.50       |
|SYN_T0002|SYN_D002 |Riyadh|2026-06-01T08:00:00+03:00|2026-06-01T08:11:00+03:00|19.25   |3.85       |
|SYN_T0003|SYN_D001 |Riyadh|2026-06-01T10:00:00+03:00|2026-06-01T10:14:00+03:00|20.50   |4.20       |
+---------+---------+------+-------------------------+-------------------------+--------+---

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Checks and next step</h2><p>A successful scenario has 144 Bronze trip deliveries but 72 distinct trip ids. Read <a href="PRACTICE.md">the reasoning prompts</a>, complete Lab 01 notes, then use the same successful workspace in <a href="STUDENT.ipynb">Lab 02</a>. No generated table is automatically committed to Git.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>الفحوص والخطوة التالية</h2><p>يحتوي السيناريو الناجح 144 سجل وصول في Bronze و72 معرف رحلة فريدًا. اقرأ <a href="PRACTICE.md">أسئلة التفكير</a> وأكمل ملاحظات اللاب 01، ثم استخدم مساحة العمل الناجحة نفسها في <a href="STUDENT.ipynb">اللاب 02</a>. لا يضاف أي جدول مولد إلى Git تلقائيًا.</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Lab 02 preparation · Cost arithmetic</h1><p>Compare two operation policies using hypothetical teaching units (TU). This is not a quotation, currency forecast or Spark runtime benchmark.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>تحضير اللاب 02 · حساب التكلفة</h1><p>قارن سياستي تشغيل بوحدات تعليم افتراضية TU. هذا ليس عرض سعر أو توقع عملة أو قياسًا لأداء Spark.</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Goal and setup</h2><p>Show how work hours, startup and overhead affect the result; do not assume scheduled compute is always cheaper. Shared code is in <code>src/masar/cost.py</code>.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>الهدف والإعداد</h2><p>بيّن أثر ساعات العمل والبدء والتكاليف الإضافية؛ ولا تفترض أن الحوسبة المجدولة أرخص دائمًا. الكود المشترك في <code dir="ltr">src/masar/cost.py</code>.</p></td></tr></tbody></table>

In [11]:
from pathlib import Path
import sys, json, tempfile
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src" / "masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the complete course repository.")
sys.path.insert(0, str(ROOT / "src"))
(ROOT / "outputs").mkdir(exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix="day01_", dir=ROOT / "outputs"))
print("Inspect the source and assumptions before running the next native section.")

Inspect the source and assumptions before running the next native section.


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>1. Inspect assumptions</h2><p>The base horizon is one 30-day teaching month. The storage charge is a monthly assumption; keep the horizon fixed when interpreting this exercise.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>1. افحص الافتراضات</h2><p>الفترة الأساسية شهر تعليمي من 30 يومًا. تكلفة التخزين افتراض شهري؛ حافظ على هذه الفترة عند تفسير التمرين.</p></td></tr></tbody></table>

In [12]:
from decimal import Decimal
from masar.cost import DEFAULTS, evaluate_cost, serializable, cost_report
print(json.dumps(DEFAULTS, indent=2))
result = cost_report()
print(json.dumps(result["base"], indent=2))

{
  "days": "30",
  "hours_per_day": "24",
  "cores": "4",
  "price_per_core_hour": "0.50",
  "storage_gib": "100",
  "storage_price_per_gib_month": "0.20",
  "work_hours_per_day": "2",
  "startup_hours_per_day": "0.25",
  "scheduled_extra_monthly": "30",
  "always_on_extra_monthly": "0"
}
{
  "unit": "TU (hypothetical teaching units; not currency)",
  "always_on_hours": "720",
  "scheduled_hours": "67.50",
  "always_on_compute": "1440.00",
  "scheduled_compute": "135.0000",
  "storage_each": "20.00",
  "always_on_total": "1460.00",
  "scheduled_total": "185.0000",
  "difference": "1275.0000",
  "reduction_fraction": "0.8732876712328767123287671233",
  "break_even_work_hours_per_day": "23.25"
}


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>2. Test workload sensitivity</h2><p>Only work hours change below. The remaining assumptions stay fixed so the comparison is interpretable.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>2. اختبر حساسية حمل العمل</h2><p>تتغير ساعات العمل فقط أدناه؛ وتبقى بقية الافتراضات ثابتة لتكون المقارنة قابلة للتفسير.</p></td></tr></tbody></table>

In [13]:
print("Work h/day | Always-on TU | Scheduled TU | Difference TU")
for row in result["sensitivity"]:
    print(f"{row['work_hours_per_day']:>10} | {row['always_on_total']:>12} | {row['scheduled_total']:>12} | {row['difference']:>13}")

Work h/day | Always-on TU | Scheduled TU | Difference TU
         0 |      1460.00 |        50.00 |       1410.00
         1 |      1460.00 |     125.0000 |     1335.0000
         2 |      1460.00 |     185.0000 |     1275.0000
         4 |      1460.00 |     305.0000 |     1155.0000
         8 |      1460.00 |     545.0000 |      915.0000
        12 |      1460.00 |     785.0000 |      675.0000
        18 |      1460.00 |    1145.0000 |      315.0000
        23 |      1460.00 |    1445.0000 |       15.0000
     23.25 |      1460.00 |    1460.0000 |        0.0000
     23.75 |      1460.00 |    1490.0000 |      -30.0000


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>3. Check boundaries and invalid inputs</h2><p>Test equality, a counterexample, zero rates and a rejected invalid input. Passing arithmetic does not establish cloud economics or real performance.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>3. افحص الحدود والمدخلات غير الصالحة</h2><p>اختبر التعادل والمثال المضاد والمعدل الصفري ورفض مدخل غير صالح. نجاح الحساب لا يثبت اقتصاديات سحابية أو أداءً فعليًا.</p></td></tr></tbody></table>

In [14]:
base = evaluate_cost(DEFAULTS)
checks = {
    "base_totals": (base["always_on_total"], base["scheduled_total"]) == (Decimal("1460"), Decimal("185")),
    "break_even": evaluate_cost({**DEFAULTS, "work_hours_per_day": "23.25"})["difference"] == 0,
    "counterexample": evaluate_cost({**DEFAULTS, "work_hours_per_day": "23.75"})["difference"] < 0,
    "zero_rate": evaluate_cost({**DEFAULTS, "price_per_core_hour": "0"})["break_even_work_hours_per_day"] is None,
}
try:
    evaluate_cost({**DEFAULTS, "cores": "-1"})
except ValueError:
    checks["negative_input_rejected"] = True
else:
    checks["negative_input_rejected"] = False
if not all(checks.values()):
    raise AssertionError(checks)
result["checks"] = checks
print(json.dumps(checks, indent=2))

{
  "base_totals": true,
  "break_even": true,
  "counterexample": true,
  "zero_rate": true,
  "negative_input_rejected": true
}


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>4. Save evidence</h2><p>The artifact explicitly records exclusions. Keep real Spark measurements separate when the full lab is available.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>4. احفظ الأدلة</h2><p>يسجل الملف ما يستبعده الحساب صراحة. افصل قياسات Spark الفعلية عندما يتاح اللاب الكامل.</p></td></tr></tbody></table>

In [15]:
output = RUN / "cost_model_result.json"
output.write_text(json.dumps(result, sort_keys=True, indent=2) + "\n", encoding="utf-8")
print("PASS: hypothetical cost arithmetic only")
print("Saved:", output.name)
print("Cost arithmetic complete. Continue to the measured Spark comparison.")

PASS: hypothetical cost arithmetic only
Saved: cost_model_result.json
Cost arithmetic complete. Continue to the measured Spark comparison.


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Interpretation and next step</h2><p>Explain the break-even workload and one excluded cost. Do not label TU as SAR or claim a measured speed-up. Continue with the Lab 02 contract when the actual benchmark is verified.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>التفسير والخطوة التالية</h2><p>اشرح حمل التعادل وتكلفة مستبعدة واحدة. لا تسم وحدات TU ريالات ولا تدع تسارعًا مقاسًا. أكمل مواصفات اللاب 02 عند التحقق من القياس الفعلي.</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Lab 02 · Observe actual Spark scans</h1><p>SDA-DSC-214 · Meaad Al-Marri</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>اللاب 02 · لاحظ قياس Spark الفعلي</h1><p>SDA-DSC-214 · ميعاد المري</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Setup</h2><p>Inspect the actual environment before accessing a prior workspace.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>الإعداد</h2><p>افحص البيئة الفعلية قبل الوصول إلى مساحة عمل سابقة.</p></td></tr></tbody></table>

In [16]:
from pathlib import Path
import sys, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from within the complete course repository")
sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.runtime import inspect_environment, require_environment, start_spark
print(json.dumps(inspect_environment(), indent=2))

{
  "scope": "DEPENDENCY_PREFLIGHT_ONLY",
  "python": "3.11.13",
  "java": "openjdk version \"17.0.20\" 2026-07-21",
  "java_major": 17,
  "packages": {
    "pyspark": {
      "required": "3.5.8",
      "observed": "3.5.8"
    },
    "delta-spark": {
      "required": "3.3.3",
      "observed": "3.3.3"
    },
    "py4j": {
      "required": "0.10.9.9",
      "observed": "0.10.9.9"
    }
  },
  "status": "DEPENDENCIES_PRESENT_ENGINE_NOT_TESTED",
  "issues": [],
  "engine_executed": false
}


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Reuse the successful project state</h2><p>No source regeneration, table overwrite or new independent project.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>استخدم حالة المشروع الناجحة</h2><p>لا إعادة توليد للمصدر ولا استبدال للجداول ولا مشروع جديد مستقل.</p></td></tr></tbody></table>

In [17]:
require_environment()
from masar.workspace import completed_bronze_workspace, require_fixed_dataset
require_fixed_dataset(SOURCE)
WORK = completed_bronze_workspace(ROOT)
spark = start_spark(WORK)
print("Workspace:", WORK.relative_to(ROOT))

Workspace: outputs/day01_bronze_jmbv27q9


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Measure equal-result actions</h2><p>CSV versus Delta version 0; one warm-up each, four measurements each in balanced order. Shared code retains the raw measurements and actual query plans. Session startup and ingestion are outside the measured interval.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>قس أفعالًا متساوية النتائج</h2><p>CSV مقابل نسخة Delta رقم صفر؛ تهيئة لكل مسار وأربعة قياسات لكل مسار بترتيب متوازن. يحفظ الكود القياسات الخام وخطط الاستعلام الفعلية. إقلاع الجلسة والاستيعاب خارج الفترة المقاسة.</p></td></tr></tbody></table>

In [18]:
from masar.benchmark import benchmark
try:
    report = benchmark(spark, SOURCE, WORK, repetitions=4)
    print(json.dumps(report["expected_and_observed_aggregate"], indent=2))
    print(json.dumps(report["measurements"], indent=2))
    print("Plans:", report["plans"])
    print("Evidence:", (WORK / "reports/benchmark.json").relative_to(ROOT))
finally:
    spark.stop()

{
  "rows": 72,
  "nonnull_fares": 72,
  "fare_total": "1794.60"
}
{
  "csv": {
    "samples_s": [
      0.484366528999999,
      0.21885121500008609,
      0.20546150999985002,
      0.40399190800008
    ],
    "median_s": 0.31142156150008304,
    "min_s": 0.20546150999985002,
    "max_s": 0.484366528999999
  },
  "delta_v0": {
    "samples_s": [
      2.461808149000035,
      1.6056460790000529,
      1.8050037470000007,
      2.9957363139999416
    ],
    "median_s": 2.133405948000018,
    "min_s": 1.6056460790000529,
    "max_s": 2.9957363139999416
  }
}
Plans: {'csv': 'reports/plans/csv.txt', 'delta_v0': 'reports/plans/delta_v0.txt'}
Evidence: outputs/day01_bronze_jmbv27q9/reports/benchmark.json


<table dir="ltr" width="100%"><thead><tr><th width="50%" dir="ltr" lang="en" align="left">English</th><th width="50%" dir="rtl" lang="ar" align="right">العربية</th></tr></thead><tbody><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Interpret and retain</h2><p>Keep all samples and explain variability. Repeated reads may use OS/JVM/metadata caches; this is not a cold-cache or production test. Complete <a href="../templates/BENCHMARKS.md">BENCHMARKS.md</a> and Lab 02 notes. See <a href="COMPLETION.md">the Day 1 handoff</a>.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>فسر واحتفظ</h2><p>احفظ جميع العينات وفسر التفاوت. قد تستفيد القراءات المتكررة من ذاكرة نظام التشغيل وJVM والبيانات الوصفية؛ ليست تجربة ذاكرة فارغة أو اختبار إنتاج. أكمل <a href="../templates/BENCHMARKS.md">BENCHMARKS.md</a> وملاحظات اللاب 02. راجع <a href="COMPLETION.md">تسليم اليوم الأول</a>.</p></td></tr></tbody></table>

<table dir="ltr" width="100%"><tr><td width="50%" valign="top" dir="ltr" lang="en" align="left"><h2>Save the handoff</h2><p>Keep the Delta tables, reports and executed notebook for Day 2. Complete the learning notes linked from project/SUBMISSION.md; there is no extra final project.</p></td><td width="50%" valign="top" dir="rtl" lang="ar" align="right"><h2>احفظ مخرجات الانتقال</h2><p>احتفظ بجداول Delta والتقارير والدفتر المنفذ لليوم الثاني. أكمل ملاحظات التعلم في project/SUBMISSION.md؛ لا يوجد مشروع نهائي إضافي.</p></td></tr></table>



In [19]:
# DAY01_HANDOFF_V2: retain the pointer and all small reports as well as Delta files.
from pathlib import Path
import zipfile
from masar.workspace import completed_bronze_workspace
WORK = completed_bronze_workspace(ROOT)
pointer = ROOT / 'outputs/day01_bronze_success.json'
files_to_save = {pointer, *(p for p in WORK.rglob('*') if p.is_file())}
for name in ('source_inspection.json', 'cost_model_result.json'):
    files_to_save.update((ROOT / 'outputs').rglob(name))
archive = ROOT / 'outputs/day01_handoff.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(files_to_save):
        bundle.write(path, arcname=path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
    assert bundle.read('outputs/day01_bronze_success.json') == pointer.read_bytes()
    assert any(name.endswith('source_inspection.json') for name in bundle.namelist())
    assert any(name.endswith('cost_model_result.json') for name in bundle.namelist())
print('Keep this ZIP for the next day:', archive)
print('Also save this notebook with outputs and your LAB01/LAB02 notes.')
if IS_COLAB:
    from google.colab import files
    files.download(str(archive))

Keep this ZIP for the next day: /content/masar-modern-data-engineering/outputs/day01_handoff.zip
Also save this notebook with outputs and your LAB01/LAB02 notes.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Day 2 · ELT and Silver</h1><p>Normalize types, cities and time zones; join safely; deduplicate before MERGE; retain late arrivals; build and test the same transformation graph with dbt.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>اليوم 2 · التحويل وبناء Silver</h1><p>وحّد الأنواع والمدن والتوقيتات، واربط البيانات دون مضاعفتها، وأزل التكرار قبل MERGE، واحتفظ بالوصول المتأخر، وابنِ التحويلات نفسها واختبرها باستخدام dbt.</p></td></tr></table>



<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>1. Continue your project</h2><p>Use the same repository and successful Day 1 workspace. Restore your handoff ZIP at the repository root when using a new session. Read <a href="README.md">today’s guide</a> before running all cells in order.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>١. استكمل مشروعك</h2><p>استخدم المستودع نفسه ومساحة اليوم الأول الناجحة. استعد ملف الانتقال في جذر المستودع عند استخدام جلسة جديدة. اقرأ <a href="README.md">دليل اليوم</a> ثم شغّل الخلايا بالترتيب.</p></td></tr></table>



In [20]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))


Continue workspace: outputs/day01_bronze_jmbv27q9


<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>03a · Prepare typed staging</h2><p>Follow <a href="labs/lab03/WALKTHROUGH.md">the lab walkthrough</a>. The cell runs real operations and saves reports; inspect the checks and explain one observation in your lab notes.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>03a · جهّز البيانات المهيأة</h2><p>اتبع <a href="labs/lab03/WALKTHROUGH.md">شرح اللاب</a>. تنفذ الخلية العمليات وتحفظ تقاريرها؛ افحص النتائج وفسّر ملاحظة واحدة في ملف اللاب.</p></td></tr></table>



In [21]:
from masar.silver import run_staging_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_staging_lab(spark, SOURCE, WORK)
    validate_stage_result('lab03a_staging', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('Observed staging row counts:', result['counts'])
    preview = WORK / ('mini_lakehouse/staging/day02_' + result['run_id'] + '/stg_trips')
    spark.read.format('delta').load(str(preview)).select('trip_id', 'city', 'fare_sar').orderBy('trip_id').show(5, truncate=False)
finally:
    spark.stop()


{
  "scope": "DAY02_STAGING_ENGINE",
  "checks": {
    "counts_verified": true,
    "typed_values_match_source_oracle": true,
    "drivers_unique_and_join_safe": true,
    "gps_valid": true,
    "delta_readback": true
  }
}
Observed staging row counts: {'stg_trips': 144, 'stg_drivers': 6, 'stg_gps': 216}
+---------+------+--------+
|trip_id  |city  |fare_sar|
+---------+------+--------+
|SYN_T0001|Riyadh|18.00   |
|SYN_T0001|Riyadh|18.00   |
|SYN_T0002|Riyadh|19.25   |
|SYN_T0002|Riyadh|19.25   |
|SYN_T0003|Riyadh|20.50   |
+---------+------+--------+
only showing top 5 rows



<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>03b · Build incremental Silver</h2><p>Follow <a href="labs/lab03/WALKTHROUGH.md">the lab walkthrough</a>. The cell runs real operations and saves reports; inspect the checks and explain one observation in your lab notes.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>03b · ابنِ Silver تدريجيًا</h2><p>اتبع <a href="labs/lab03/WALKTHROUGH.md">شرح اللاب</a>. تنفذ الخلية العمليات وتحفظ تقاريرها؛ افحص النتائج وفسّر ملاحظة واحدة في ملف اللاب.</p></td></tr></table>



In [22]:
from masar.silver import run_incremental_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_incremental_lab(spark, SOURCE, WORK)
    validate_stage_result('lab03b_silver', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    spark.read.format('delta').load(str(WORK/'mini_lakehouse/silver/trips')).select('trip_id', 'city', 'fare_sar').orderBy('trip_id').show(5, truncate=False)
finally:
    spark.stop()


{
  "scope": "DAY02_SILVER_ENGINE",
  "checks": {
    "all_scenarios_match_independent_oracle": true,
    "business_keys_unique": true,
    "replay_preserves_business_content": true,
    "late_rows_retained": true,
    "actual_delta_files": true
  }
}
+-----------+------+--------+
|trip_id    |city  |fare_sar|
+-----------+------+--------+
|SYN_LATE001|Riyadh|25.00   |
|SYN_LATE002|Jeddah|27.00   |
|SYN_LATE003|Dammam|29.00   |
|SYN_T0001  |Riyadh|18.00   |
|SYN_T0002  |Riyadh|19.25   |
+-----------+------+--------+
only showing top 5 rows



<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Lab 03 · Run the dbt models</h2><p>Read <a href="DBT_GUIDE.md">the six-model graph and tests</a>. The session adapter uses isolated copies of Bronze; it does not replace your Silver workspace.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>Lab 03 · شغّل نماذج dbt</h2><p>اقرأ <a href="DBT_GUIDE.md">تسلسل النماذج الستة واختباراتها</a>. يستخدم موصل الجلسة نسخًا معزولة من Bronze ولا يستبدل مساحة Silver الخاصة بك.</p></td></tr></table>



In [23]:
%pip install dbt-core==1.9.8 dbt-spark==1.9.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.4/114.4 kB 5.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of dbt-adapters to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of dbt-adapters to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of dbt-common to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of dbt-common to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dep

In [24]:
from masar.dbt_lab import run_dbt_lab
dbt_report, dbt_path = run_dbt_lab(ROOT)
print(json.dumps({'status': dbt_report['status'], 'phases_completed': len(dbt_report['phases']), 'report': str(dbt_path.relative_to(ROOT)), 'error': dbt_report.get('error')}, indent=2))
assert dbt_report['status'] == 'PASSED_DBT_NATIVE', dbt_report.get('error')

# Observed learning output
for phase in dbt_report['phases']:
    print(phase['phase'], 'rows:', phase['rows'], 'fare SAR:', phase['total_fare_sar'])
print('Catalog evidence:', dbt_report['commands'][-1])


{
  "status": "PASSED_DBT_NATIVE",
  "phases_completed": 4,
  "report": "outputs/dbt_validation_nh58vl02/reports/dbt_attempt.json",
  "error": null
}
base rows: 72 fare SAR: 1794.60
rerun rows: 72 fare SAR: 1794.60
late rows: 75 fare SAR: 1875.60
late_replay rows: 75 fare SAR: 1875.60
Catalog evidence: {'catalog_sha256': '6ab271bfc355c069d448a4880ce18d34530b414402cdf25e769aa421d536bb65', 'models_documented': 6, 'sources_documented': 3, 'metadata_method': 'native DESCRIBE TABLE EXTENDED', 'phase': 'documentation', 'command': ['docs', 'generate'], 'target': 'dbt/commands/documentation/target'}


<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Review and save</h2><p>Answer <a href="PRACTICE.md">the questions</a> as part of the existing lab notes, then use <a href="COMPLETION.md">the completion checklist</a>. Save your notebook with actual outputs. The archive below is a handoff, not another assignment.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>راجع واحفظ</h2><p>أجب عن <a href="PRACTICE.md">الأسئلة</a> ضمن ملاحظات اللاب، ثم استخدم <a href="COMPLETION.md">قائمة الاكتمال</a>. احفظ دفترك بالمخرجات الفعلية. ملف الانتقال أدناه ليس تكليفًا آخر.</p></td></tr></table>



In [25]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day02_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    dbt_workspace = dbt_path.parent.parent
    for p in sorted(dbt_workspace.rglob('*')):
        if p.is_file():
            bundle.write(p, p.relative_to(ROOT).as_posix())
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))


Retain the notebook outputs, notes and outputs/day02_handoff.zip


<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Day 3 · Delta transactions and maintenance</h1><p>Apply a correction once; preserve revision precedence; read earlier versions; test schema changes and maintenance on isolated copies.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>اليوم 3 · معاملات Delta والصيانة</h1><p>طبّق التصحيح مرة واحدة، واحفظ أولوية المراجعات، واقرأ النسخ السابقة، واختبر تغيير المخطط والصيانة على نسخ معزولة.</p></td></tr></table>



<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>1. Continue your project</h2><p>Use the same repository and successful Day 1 workspace. Restore your handoff ZIP at the repository root when using a new session. Read <a href="README.md">today’s guide</a> before running all cells in order.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>١. استكمل مشروعك</h2><p>استخدم المستودع نفسه ومساحة اليوم الأول الناجحة. استعد ملف الانتقال في جذر المستودع عند استخدام جلسة جديدة. اقرأ <a href="README.md">دليل اليوم</a> ثم شغّل الخلايا بالترتيب.</p></td></tr></table>



In [26]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))


Continue workspace: outputs/day01_bronze_jmbv27q9


<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>04a · Correct and read versions</h2><p>Follow <a href="labs/lab04/WALKTHROUGH.md">the lab walkthrough</a>. The cell runs real operations and saves reports; inspect the checks and explain one observation in your lab notes.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>04a · صحّح واقرأ النسخ</h2><p>اتبع <a href="labs/lab04/WALKTHROUGH.md">شرح اللاب</a>. تنفذ الخلية العمليات وتحفظ تقاريرها؛ افحص النتائج وفسّر ملاحظة واحدة في ملف اللاب.</p></td></tr></table>



In [27]:
from masar.delta_lab import run_transactions_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_transactions_lab(spark, SOURCE, WORK)
    validate_stage_result('lab04a_transactions', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print(json.dumps({k: result[k] for k in ('before','after_correction','past_version_read')}, indent=2, default=str))
finally:
    spark.stop()


{
  "scope": "DAY03_TRANSACTIONS_ENGINE",
  "checks": {
    "native_correction_matches_source_expectation": true,
    "business_rows_stay_75": true,
    "replay_and_stale_delivery_preserve_values": true,
    "same_revision_conflict_rejected": true,
    "actual_prior_version_read": true,
    "mixed_valid_invalid_batch_rejected_atomically": true
  }
}
{
  "before": {
    "rows": 75,
    "business_digest": "0d16e2795620ae0c0f54d3fcd52a5b13fb2c2a47cd4d0c42c8ddb195f19bc6e0",
    "version": 1,
    "schema": [
      [
        "trip_id",
        "string"
      ],
      [
        "driver_id",
        "string"
      ],
      [
        "city",
        "string"
      ],
      [
        "start_utc",
        "timestamp"
      ],
      [
        "end_utc",
        "timestamp"
      ],
      [
        "trip_date_local",
        "date"
      ],
      [
        "fare_sar",
        "decimal(12,2)"
      ],
      [
        "distance_km",
        "decimal(12,2)"
      ],
      [
        "duration_seconds",

<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>04b · Test safe maintenance</h2><p>Follow <a href="labs/lab04/WALKTHROUGH.md">the lab walkthrough</a>. The cell runs real operations and saves reports; inspect the checks and explain one observation in your lab notes.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>04b · اختبر الصيانة الآمنة</h2><p>اتبع <a href="labs/lab04/WALKTHROUGH.md">شرح اللاب</a>. تنفذ الخلية العمليات وتحفظ تقاريرها؛ افحص النتائج وفسّر ملاحظة واحدة في ملف اللاب.</p></td></tr></table>



In [28]:
from masar.delta_lab import run_maintenance_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_maintenance_lab(spark, SOURCE, WORK)
    validate_stage_result('lab04b_maintenance', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print(json.dumps({'recovery': result['recovery'], 'vacuum': result['vacuum']}, indent=2, default=str))
finally:
    spark.stop()


{
  "scope": "DAY03_MAINTENANCE_ENGINE",
  "checks": {
    "unexpected_column_rejected": true,
    "approved_evolution_preserves_business_values": true,
    "compaction_preserves_values": true,
    "delete_affects_copy_only": true,
    "restore_creates_new_commit": true,
    "vacuum_is_non_destructive_dry_run": true,
    "trusted_silver_unchanged": true
  }
}
{
  "recovery": {
    "before": {
      "rows": 75,
      "business_digest": "1321d375742d806a1f0fef82be9e2862af04f5d18f3a976792a6b1d9aa1b4383",
      "version": 0,
      "schema": [
        [
          "trip_id",
          "string"
        ],
        [
          "driver_id",
          "string"
        ],
        [
          "city",
          "string"
        ],
        [
          "start_utc",
          "timestamp"
        ],
        [
          "end_utc",
          "timestamp"
        ],
        [
          "trip_date_local",
          "date"
        ],
        [
          "fare_sar",
          "decimal(12,2)"
        ],
       

<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Review and save</h2><p>Answer <a href="PRACTICE.md">the questions</a> as part of the existing lab notes, then use <a href="COMPLETION.md">the completion checklist</a>. Save your notebook with actual outputs. The archive below is a handoff, not another assignment.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>راجع واحفظ</h2><p>أجب عن <a href="PRACTICE.md">الأسئلة</a> ضمن ملاحظات اللاب، ثم استخدم <a href="COMPLETION.md">قائمة الاكتمال</a>. احفظ دفترك بالمخرجات الفعلية. ملف الانتقال أدناه ليس تكليفًا آخر.</p></td></tr></table>



In [29]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day03_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))


Retain the notebook outputs, notes and outputs/day03_handoff.zip


<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Day 4 · Streaming, quality and governance</h1><p>Receive Kafka events with a persistent checkpoint; reconcile delivery and event identities; validate, quarantine and recheck a batch; document quality and governance decisions.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>اليوم 4 · التدفق والجودة والحوكمة</h1><p>استقبل أحداث Kafka مع نقطة تحقق مستمرة، وطابق سجلات الوصول وهويات الأحداث، وافحص الدفعة واعزل المعيب وأعد الفحص، ووثّق قرارات الجودة والحوكمة.</p></td></tr></table>



<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>1. Continue your project</h2><p>Use the same repository and successful Day 1 workspace. Restore your handoff ZIP at the repository root when using a new session. Read <a href="README.md">today’s guide</a> before running all cells in order.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>١. استكمل مشروعك</h2><p>استخدم المستودع نفسه ومساحة اليوم الأول الناجحة. استعد ملف الانتقال في جذر المستودع عند استخدام جلسة جديدة. اقرأ <a href="README.md">دليل اليوم</a> ثم شغّل الخلايا بالترتيب.</p></td></tr></table>



In [30]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))


Continue workspace: outputs/day01_bronze_jmbv27q9


<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>05 · Receive Kafka events</h2><p>Follow <a href="labs/lab05/WALKTHROUGH.md">the lab walkthrough</a>. The cell runs real operations and saves reports; inspect the checks and explain one observation in your lab notes.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>05 · استقبل أحداث Kafka</h2><p>اتبع <a href="labs/lab05/WALKTHROUGH.md">شرح اللاب</a>. تنفذ الخلية العمليات وتحفظ تقاريرها؛ افحص النتائج وفسّر ملاحظة واحدة في ملف اللاب.</p></td></tr></table>



In [31]:
from pathlib import Path
import os, sys, subprocess, socket, time

ROOT = Path("/content/masar-modern-data-engineering")

if not (ROOT / "course.json").is_file():
    raise RuntimeError(
        "ملفات المشروع غير موجودة. افتح جلسة Colab التي عملت فيها على الأيام السابقة."
    )

os.chdir(ROOT)

print("1/3 تثبيت متطلبات اليوم الرابع...", flush=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "-r", str(ROOT / "requirements-day04.txt")
    ],
    check=True
)

from kafka.admin import KafkaAdminClient

BASE = Path("/content/masar-kafka-local")
HOME_KAFKA = BASE / "kafka_2.13-4.0.2"
DATA = BASE / "data"
CONFIG = BASE / "server.properties"
LOG = BASE / "server.log"

BASE.mkdir(exist_ok=True)

def port_open():
    try:
        with socket.create_connection(("127.0.0.1", 9092), timeout=1):
            return True
    except OSError:
        return False

print("2/3 تجهيز Kafka داخل جلسة Colab...", flush=True)

if not port_open():
    if not (HOME_KAFKA / "bin/kafka-server-start.sh").is_file():
        archive = BASE / "kafka.tgz"

        subprocess.run(
            [
                "curl", "-fL", "--retry", "2",
                "-o", str(archive),
                "https://archive.apache.org/dist/kafka/4.0.2/kafka_2.13-4.0.2.tgz"
            ],
            check=True
        )

        subprocess.run(
            ["tar", "-xzf", str(archive), "-C", str(BASE)],
            check=True
        )

    CONFIG.write_text(
        f"""process.roles=broker,controller
node.id=1
controller.quorum.bootstrap.servers=127.0.0.1:9093
listeners=PLAINTEXT://127.0.0.1:9092,CONTROLLER://127.0.0.1:9093
advertised.listeners=PLAINTEXT://127.0.0.1:9092
controller.listener.names=CONTROLLER
inter.broker.listener.name=PLAINTEXT
listener.security.protocol.map=CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
log.dirs={DATA}
num.partitions=1
offsets.topic.replication.factor=1
transaction.state.log.replication.factor=1
transaction.state.log.min.isr=1
group.initial.rebalance.delay.ms=0
""",
        encoding="utf-8"
    )

    env = os.environ.copy()
    env["KAFKA_HEAP_OPTS"] = "-Xms256m -Xmx512m"
    storage = str(HOME_KAFKA / "bin/kafka-storage.sh")

    if not (DATA / "meta.properties").exists():
        if DATA.exists() and any(DATA.iterdir()):
            raise RuntimeError(
                "توجد بيانات Kafka سابقة غير مكتملة. أرسل هذه الرسالة قبل المتابعة."
            )

        cluster = subprocess.check_output(
            [storage, "random-uuid"],
            env=env,
            text=True
        ).strip()

        subprocess.run(
            [
                storage, "format", "--standalone",
                "-t", cluster, "-c", str(CONFIG)
            ],
            env=env,
            check=True
        )

    with LOG.open("a") as output:
        process = subprocess.Popen(
            [
                str(HOME_KAFKA / "bin/kafka-server-start.sh"),
                str(CONFIG)
            ],
            stdout=output,
            stderr=subprocess.STDOUT,
            env=env,
            start_new_session=True
        )

print("3/3 التحقق من الاتصال...", flush=True)

ready = False

for attempt in range(30):
    try:
        client = KafkaAdminClient(
            bootstrap_servers="127.0.0.1:9092",
            api_version_auto_timeout_ms=3000,
            request_timeout_ms=10000
        )
        try:
            client.list_topics()
        finally:
            client.close()

        ready = True
        break

    except Exception:
        time.sleep(2)

if not ready:
    if LOG.exists():
        print(LOG.read_text(errors="replace")[-6000:])

    raise RuntimeError(
        "لم يكتمل تشغيل Kafka. أرسل آخر رسالة ظهرت في هذه الخلية."
    )

subprocess.run(
    [sys.executable, "scripts/run_day04.py", "--preflight"],
    check=True
)

print("✅ اكتمل فحص الإعداد. شغّل الآن خلية التمرين الموجودة أسفل هذه الخلية.")

1/3 تثبيت متطلبات اليوم الرابع...
2/3 تجهيز Kafka داخل جلسة Colab...
3/3 التحقق من الاتصال...


ERROR:kafka.conn:<BrokerConnection client_id=kafka-python-2.2.15, node_id=bootstrap-0 host=127.0.0.1:9092 <connecting> [IPv4 ('127.0.0.1', 9092)]>: Connect attempt returned error 111. Disconnecting.
ERROR:kafka.conn:<BrokerConnection client_id=kafka-python-2.2.15, node_id=bootstrap-0 host=127.0.0.1:9092 <connecting> [IPv4 ('127.0.0.1', 9092)]>: Closing connection. KafkaConnectionError: 111 ECONNREFUSED
ERROR:kafka.conn:<BrokerConnection client_id=kafka-python-2.2.15, node_id=bootstrap-0 host=127.0.0.1:9092 <connecting> [IPv4 ('127.0.0.1', 9092)]>: Connect attempt returned error 111. Disconnecting.
ERROR:kafka.conn:<BrokerConnection client_id=kafka-python-2.2.15, node_id=bootstrap-0 host=127.0.0.1:9092 <connecting> [IPv4 ('127.0.0.1', 9092)]>: Closing connection. KafkaConnectionError: 111 ECONNREFUSED
ERROR:kafka.conn:<BrokerConnection client_id=kafka-python-2.2.15, node_id=bootstrap-0 host=127.0.0.1:9092 <connecting> [IPv4 ('127.0.0.1', 9092)]>: Connect attempt returned error 111. Disc

✅ اكتمل فحص الإعداد. شغّل الآن خلية التمرين الموجودة أسفل هذه الخلية.


In [32]:
from pathlib import Path
import os
import sys
import subprocess

ROOT = Path("/content/masar-modern-data-engineering")

if "SOURCE" not in globals() or "WORK" not in globals():
    raise RuntimeError(
        "شغّل أول خلية إعداد في دفتر اليوم الرابع، ثم أعد تشغيل هذه الخلية."
    )

# تحميل مكتبات Spark المطلوبة منذ بداية التشغيل
env = os.environ.copy()

for name in ("PYSPARK_GATEWAY_PORT", "PYSPARK_GATEWAY_SECRET"):
    env.pop(name, None)

env["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "io.delta:delta-spark_2.12:3.3.3,"
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.8 "
    "pyspark-shell"
)

code = r'''
from pathlib import Path
import sys
import json
import pyspark

ROOT = Path(sys.argv[1])
SOURCE = Path(sys.argv[2])
WORK = Path(sys.argv[3])

sys.path.insert(0, str(ROOT / "src"))

if pyspark.__version__ != "3.5.8":
    raise RuntimeError(
        "إصدار Spark مختلف عن إصدار الدورة: " + pyspark.__version__
    )

from masar.runtime import start_spark
from masar.native_contracts import validate_stage_result
from kafka.admin import KafkaAdminClient
import masar.streaming as streaming

print("بدء Spark وتحميل مكتبة Kafka...", flush=True)
spark = start_spark(WORK, kafka=True)

try:
    # فحص وجود موصل Kafka دون تشغيل تدفق أو نشر رسائل
    probe = (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", "127.0.0.1:9092")
        .option("subscribePattern", ".*")
        .load()
    )

    print("✅ تم تحميل موصل Kafka.", flush=True)

    # حماية رسائل المحاولات السابقة من إعادة الإرسال
    admin = KafkaAdminClient(
        bootstrap_servers="127.0.0.1:9092",
        api_version_auto_timeout_ms=5000,
        request_timeout_ms=10000
    )
    try:
        previous_topics = set(admin.list_topics())
    finally:
        admin.close()

    original_publish = streaming.publish_fixture

    def guarded_publish(source, topic, *args, **kwargs):
        if topic in previous_topics:
            raise RuntimeError(
                "توقّف التمرين لحماية رسائل محاولة سابقة. "
                "أرسل هذه الرسالة دون حذف أي ملفات. Topic: " + topic
            )
        return original_publish(source, topic, *args, **kwargs)

    streaming.publish_fixture = guarded_publish

    print("تشغيل محاولة جديدة للتمرين...", flush=True)

    result = streaming.run_stream_lab(spark, SOURCE, WORK)
    validate_stage_result("lab05_streaming", result)

    print(json.dumps({
        "scope": result["scope"],
        "checks": result["checks"]
    }, indent=2))

    print(
        "Transport rows:",
        [phase["transport_rows"] for phase in result["phases"]]
    )

    print(
        "Unique event IDs:",
        [phase["unique_event_ids"] for phase in result["phases"]]
    )

    (
        spark.read.format("delta")
        .load(str(WORK / result["event_table"]))
        .select("event_id", "trip_id", "event_ts")
        .orderBy("event_id")
        .show(5, truncate=False)
    )

finally:
    spark.stop()
'''

print("جارٍ تحميل المكتبات وتشغيل التمرين؛ انتظر انتهاء الخلية.", flush=True)

process = subprocess.Popen(
    [
        sys.executable, "-u", "-c", code,
        str(ROOT), str(Path(SOURCE).resolve()), str(Path(WORK).resolve())
    ],
    cwd=str(ROOT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end="", flush=True)

if process.wait() != 0:
    raise RuntimeError(
        "لم يكتمل التمرين. أرسل آخر 20 سطرًا ظهرت فوق هذه الرسالة."
    )

print("✅ اكتمل التمرين بنجاح. يمكنك تشغيل الخلية التالية.")

جارٍ تحميل المكتبات وتشغيل التمرين؛ انتظر انتهاء الخلية.
بدء Spark وتحميل مكتبة Kafka...
https://repo.maven.apache.org/maven2 added as a remote repository with the name: repo-1
:: loading settings :: url = jar:file:/usr/local/lib/python3.11/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-71d452f2-90ed-4fb9-8a2f-7c6284b0b42d;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.3 in central
	found io.delta#delta-storage;3.3.3 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.8 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.8 in central
	found org.apache.kafka#kafka-clients;3.4.1 i

<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>06 · Validate and quarantine</h2><p>Follow <a href="labs/lab06/WALKTHROUGH.md">the lab walkthrough</a>. The cell runs real operations and saves reports; inspect the checks and explain one observation in your lab notes.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>06 · افحص واعزل السجلات</h2><p>اتبع <a href="labs/lab06/WALKTHROUGH.md">شرح اللاب</a>. تنفذ الخلية العمليات وتحفظ تقاريرها؛ افحص النتائج وفسّر ملاحظة واحدة في ملف اللاب.</p></td></tr></table>



In [3]:
from google.colab import files
from pathlib import Path
import hashlib, io, zipfile, subprocess, sys

uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError("اختر ملف masar_colab_recovery.zip فقط.")

data = next(iter(uploaded.values()))
expected = "ffe0748aebeee1e00c07db6abb9be1022b47c932d27a12915d3c1d02e5df3d58"

if hashlib.sha256(data).hexdigest() != expected:
    raise RuntimeError("الملف مختلف. اختر ملف الاستعادة المرفق في المحادثة.")

folder = Path("/content/masar_recovery")
folder.mkdir(exist_ok=True)

with zipfile.ZipFile(io.BytesIO(data)) as archive:
    archive.extractall(folder)

process = subprocess.Popen(
    [sys.executable, "-u", str(folder / "restore_masar.py")],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end="", flush=True)

if process.wait() != 0:
    raise RuntimeError("توقف الإعداد. أرسل آخر الرسائل الظاهرة أعلاه.")

Saving masar_colab_recovery.zip to masar_colab_recovery.zip
1/4 تنزيل كود مشروعك من GitHub...
2/4 استعادة مخرجاتك السابقة...
الملفات المستعادة: 504
3/4 تجهيز Python 3.11 وJava 17 ومتطلبات فحص الجودة...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 45.4 MB/s eta 0:00:00
Using CPython 3.11.13 interpreter at: /usr/bin/python3
Creating virtual environment with seed packages at: masar-course-py311
 + packaging==26.3
 + pip==26.2.1
 + setuptools==84.0.0
 + wheel==0.48.0
Activate with: source masar-course-py311/bin/activate
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-17-jre-headless:amd64.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading database ... 20%
(Reading database ... 25%
(Reading database ... 30%
(Reading database ... 35%
(R

In [4]:
from pathlib import Path
from google.colab import files
import subprocess

process = subprocess.Popen(
    [
        "/content/masar-course-py311/bin/python",
        "-u",
        "/content/masar_recovery/run_quality.py"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end="", flush=True)

if process.wait() != 0:
    raise RuntimeError("لم يكتمل الفحص. أرسل آخر الرسائل الظاهرة أعلاه.")

saved_zip = Path(
    "/content/masar_recovery/latest_handoff_path.txt"
).read_text().strip()

files.download(saved_zip)

تشغيل محاولة جديدة لفحص الجودة؛ انتظر انتهاء الخلية.
Python: 3.11.13
https://repo.maven.apache.org/maven2 added as a remote repository with the name: repo-1
:: loading settings :: url = jar:file:/content/masar-course-py311/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-77276f27-98c9-44f2-886a-b146db102167;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.3 in central
	found io.delta#delta-storage;3.3.3 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.3.3/delta-spark_2.12-3.3.3.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.3.3!delta-spark_2.12.jar (395ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.3.3/delt

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Review and save</h2><p>Answer <a href="PRACTICE.md">the questions</a> as part of the existing lab notes, then use <a href="COMPLETION.md">the completion checklist</a>. Save your notebook with actual outputs. The archive below is a handoff, not another assignment.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>راجع واحفظ</h2><p>أجب عن <a href="PRACTICE.md">الأسئلة</a> ضمن ملاحظات اللاب، ثم استخدم <a href="COMPLETION.md">قائمة الاكتمال</a>. احفظ دفترك بالمخرجات الفعلية. ملف الانتقال أدناه ليس تكليفًا آخر.</p></td></tr></table>



In [34]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day04_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))


Retain the notebook outputs, notes and outputs/day04_handoff.zip


<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h1>Day 5 · Gold, AI/BI and project submission</h1><p>Build and reconcile Gold tables; produce reporting and point-in-time features; keep unknown future labels null; recover from a failed candidate and submit the cumulative work.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h1>اليوم 5 · طبقة Gold ومخرجات AI وBI وتسليم المشروع</h1><p>ابنِ جداول Gold وطابق مجاميعها، وجهّز التقارير والخصائص المتاحة وقت القرار، وأبقِ القيم المستقبلية غير المعروفة فارغة، وتعافَ من فشل بناء نسخة مرشحة وسلّم العمل التراكمي.</p></td></tr></table>



<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>1. Continue your project</h2><p>Use the same repository and successful Day 1 workspace. Restore your handoff ZIP at the repository root when using a new session. Read <a href="README.md">today’s guide</a> before running all cells in order.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>١. استكمل مشروعك</h2><p>استخدم المستودع نفسه ومساحة اليوم الأول الناجحة. استعد ملف الانتقال في جذر المستودع عند استخدام جلسة جديدة. اقرأ <a href="README.md">دليل اليوم</a> ثم شغّل الخلايا بالترتيب.</p></td></tr></table>



In [2]:
from google.colab import files
from pathlib import Path
from urllib.request import urlretrieve
import hashlib, subprocess, sys

uploaded = files.upload()

if len(uploaded) != 1:
    raise RuntimeError("اختر ملف المخرجات المطلوب فقط.")

data = next(iter(uploaded.values()))
expected = "93c80093b3838ff962bd302c635789164b3765928891a6ce9898fccb979dd950"

if hashlib.sha256(data).hexdigest() != expected:
    raise RuntimeError("اختر أحدث ملف مخرجات، الموضح في المحادثة.")

folder = Path("/content/masar_recovery")
folder.mkdir(exist_ok=True)

archive = folder / "latest_handoff.zip"
archive.write_bytes(data)

base = (
    "https://raw.githubusercontent.com/"
    "sgmutairi-sys/masar-modern-data-engineering/"
    "32552a04f02289a7879beee6fcece745b1d9c181/"
    "scripts/colab_recovery/"
)

for name in ["restore_masar.py", "check_day05_setup.py"]:
    urlretrieve(base + name, str(folder / name))

process = subprocess.Popen(
    [
        sys.executable, "-u",
        str(folder / "restore_masar.py"),
        "--archive", str(archive),
        "--day05"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for line in process.stdout:
    print(line, end="", flush=True)

if process.wait() != 0:
    raise RuntimeError("أرسل آخر الرسائل الظاهرة أعلاه.")

Saving day04_quality_recovery_20260916T180618912333Z.zip to day04_quality_recovery_20260916T180618912333Z.zip
الأرشيف المعتمد: day04_quality_recovery_20260916T180618912333Z.zip
1/4 تنزيل كود مشروعك من GitHub...
2/4 استعادة مخرجاتك السابقة...
الملفات المستعادة: 631
3/4 تجهيز Python 3.11 وJava 17 ومتطلبات فحص الجودة...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 34.1 MB/s eta 0:00:00
Using CPython 3.11.13 interpreter at: /usr/bin/python3
Creating virtual environment with seed packages at: masar-course-py311
 + packaging==26.3
 + pip==26.2.1
 + setuptools==84.0.0
 + wheel==0.48.0
Activate with: source masar-course-py311/bin/activate
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-17-jre-headless:amd64.
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading databa

<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>07 · Build Gold and recover</h2><p>Follow <a href="labs/lab07/WALKTHROUGH.md">the lab walkthrough</a>. The cell runs real operations and saves reports; inspect the checks and explain one observation in your lab notes.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>07 · ابنِ Gold واختبر التعافي</h2><p>اتبع <a href="labs/lab07/WALKTHROUGH.md">شرح اللاب</a>. تنفذ الخلية العمليات وتحفظ تقاريرها؛ افحص النتائج وفسّر ملاحظة واحدة في ملف اللاب.</p></td></tr></table>



In [36]:
from masar.serving import run_recovery_exercise
spark = start_spark(WORK, kafka=False)
try:
    result = run_recovery_exercise(spark, SOURCE, WORK)
    validate_stage_result('lab07_gold_recovery', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
finally:
    spark.stop()


{
  "scope": "DAY05_NATIVE_RECOVERY",
  "checks": {
    "injected_failure_observed": true,
    "previous_release_preserved": true,
    "rebuild_has_new_identity": true,
    "content_equal": true
  }
}


<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>08 · Serve AI/BI data</h2><p>Follow <a href="labs/lab08/WALKTHROUGH.md">the lab walkthrough</a>. The cell runs real operations and saves reports; inspect the checks and explain one observation in your lab notes.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>08 · جهّز بيانات AI وBI</h2><p>اتبع <a href="labs/lab08/WALKTHROUGH.md">شرح اللاب</a>. تنفذ الخلية العمليات وتحفظ تقاريرها؛ افحص النتائج وفسّر ملاحظة واحدة في ملف اللاب.</p></td></tr></table>



In [37]:
from masar.serving import run_serving_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_serving_lab(spark, SOURCE, WORK)
    validate_stage_result('lab08_serving', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('BI totals:', json.dumps(result['bi_summary'], indent=2))
    from masar.serving import read_release
    _, observed_tables = read_release(spark, WORK)
    print('AI feature example:', observed_tables['ai.zone_hourly_features'][0])
    print('Future label example:', observed_tables['ai.zone_hourly_labels'][0])
finally:
    spark.stop()


{
  "scope": "DAY05_NATIVE_SERVING",
  "checks": {
    "gold.zone_hourly_demand_schema_and_keys": true,
    "gold.driver_daily_schema_and_keys": true,
    "bi.dim_zone_schema_and_keys": true,
    "bi.dim_driver_schema_and_keys": true,
    "bi.dim_date_schema_and_keys": true,
    "bi.fact_trips_schema_and_keys": true,
    "ai.zone_hourly_features_schema_and_keys": true,
    "ai.zone_hourly_labels_schema_and_keys": true,
    "fact_grain_75": true,
    "foreign_keys_valid": true,
    "gold_fact_totals_match": true,
    "events_aggregated_before_join": true,
    "group_grains_reconcile": true,
    "labels_not_fabricated": true,
    "feature_availability_checked": true,
    "feature_label_keys_aligned": true
  }
}
BI totals: [
  {
    "zone_key": "Z_DAMMAM",
    "trip_count": 25,
    "total_fare_sar": "670.40"
  },
  {
    "zone_key": "Z_JEDDAH",
    "trip_count": 25,
    "total_fare_sar": "625.20"
  },
  {
    "zone_key": "Z_RIYADH",
    "trip_count": 25,
    "total_fare_sar": "585.00"
  }

<table dir="ltr" width="100%"><tr><td width="50%" dir="ltr" lang="en" align="left" valign="top"><h2>Review and save</h2><p>Answer <a href="PRACTICE.md">the questions</a> as part of the existing lab notes, then use <a href="COMPLETION.md">the completion checklist</a>. Save your notebook with actual outputs. The archive below is a handoff, not another assignment.</p></td><td width="50%" dir="rtl" lang="ar" align="right" valign="top"><h2>راجع واحفظ</h2><p>أجب عن <a href="PRACTICE.md">الأسئلة</a> ضمن ملاحظات اللاب، ثم استخدم <a href="COMPLETION.md">قائمة الاكتمال</a>. احفظ دفترك بالمخرجات الفعلية. ملف الانتقال أدناه ليس تكليفًا آخر.</p></td></tr></table>



In [38]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day05_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))


Retain the notebook outputs, notes and outputs/day05_handoff.zip


In [39]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

root = Path("/content/masar-modern-data-engineering")

if not (root / "course.json").is_file():
    raise RuntimeError(
        "ملفات المشروع غير موجودة في هذه الجلسة. "
        "شغّل الخلية في جلسة عملك السابقة."
    )

selected = set()

# تعليمات المشروع وقوالبه وملاحظاته
folders = [
    "project", "templates", "docs", "labs", "notes", "decisions",
    "day01", "day02", "day03", "day04", "day05"
]

for folder in folders:
    location = root / folder
    if location.exists():
        for path in location.rglob("*.md"):
            if path.is_file():
                selected.add(path)

# تعريف المشروع ومتطلبات التشغيل
for pattern in ("*.md", "course.json", "requirements*.txt"):
    selected.update(p for p in root.glob(pattern) if p.is_file())

# تقارير النتائج الفعلية الموجودة
outputs = root / "outputs"

if outputs.exists():
    for path in outputs.rglob("*"):
        if (
            path.is_file()
            and "reports" in path.relative_to(outputs).parts
            and path.suffix.lower() in {".json", ".md", ".txt", ".csv"}
        ):
            selected.add(path)

    pointer = outputs / "day01_bronze_success.json"
    if pointer.is_file():
        selected.add(pointer)

    handoff = outputs / "day05_handoff.zip"
    if handoff.is_file():
        selected.add(handoff)
        print("تم العثور على حزمة مخرجات اليوم الخامس.")
    else:
        print("لم توجد حزمة اليوم الخامس؛ سنراجع ما اكتمل حتى الآن.")

archive = Path("/content/masar_submission_review.zip")

with ZipFile(archive, "w", ZIP_DEFLATED) as bundle:
    for path in sorted(selected):
        bundle.write(path, path.relative_to(root).as_posix())

print(f"تم جمع {len(selected)} ملفًا للمراجعة.")
files.download(str(archive))

تم العثور على حزمة مخرجات اليوم الخامس.
تم جمع 144 ملفًا للمراجعة.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [40]:
from urllib.request import urlretrieve
import runpy

urlretrieve(
    "https://raw.githubusercontent.com/sgmutairi-sys/masar-modern-data-engineering/b4d7ec369cd1b47d51e552e2fbecc54b058c87cc/scripts/collect_submission_evidence.py",
    "/content/collect_submission_evidence.py"
)

runpy.run_path(
    "/content/collect_submission_evidence.py",
    run_name="__main__"
)

وجدت: day01_handoff.zip
وجدت: day02_handoff.zip
وجدت مساحة dbt الناجحة: dbt_validation_nh58vl02
جُمعت 721 ملفات موجودة. أرفق الملف الناتج في المحادثة.
نزّل الدفاتر الخمسة من قائمة ملف في Colab وأرفقها أيضًا.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

{'__name__': '__main__',
 '__doc__': None,
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': '/content/collect_submission_evidence.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import__': <function __import__(name, globals=None, locals=None, fromlist=(), level=0)